In [511]:
import pandas as pd
import numpy as np
from pvlib.solarposition import get_solarposition
from pvlib.location import Location

### Load Data

In [512]:
df = pd.read_csv('../data/plant_weather_merged.csv', parse_dates=['DATE_TIME'])

In [513]:
df.head()

,DATE_TIME,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,0.0,0.0,0.0,6.837223e+06,25.134452,22.809588,0.0
1,2020-05-15 00:30:00,0.0,0.0,0.0,6.837223e+06,24.890942,22.476579,0.0
2,2020-05-15 01:00:00,0.0,0.0,0.0,6.845012e+06,24.578809,22.066997,0.0
3,2020-05-15 01:30:00,0.0,0.0,0.0,6.845012e+06,24.755848,22.756922,0.0
4,2020-05-15 02:00:00,0.0,0.0,0.0,6.837223e+06,24.974589,23.184671,0.0


In [514]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1632 entries, 0 to 1631
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   DATE_TIME            1632 non-null   datetime64[ns]
 1   DC_POWER             1632 non-null   float64       
 2   AC_POWER             1632 non-null   float64       
 3   DAILY_YIELD          1632 non-null   float64       
 4   TOTAL_YIELD          1632 non-null   float64       
 5   AMBIENT_TEMPERATURE  1632 non-null   float64       
 6   MODULE_TEMPERATURE   1632 non-null   float64       
 7   IRRADIATION          1632 non-null   float64       
dtypes: datetime64[ns](1), float64(7)
memory usage: 102.1 KB


### Time-Based Features

Cyclical encodings (hour_sin, hour_cos) capture the daily solar cycle without artificial discontinuities between hours 23 and 0. hour_squared (distance from noon) captures the parabolic shape of the solar production curve - output increases as we approach midday and decreases symmetrically.  

prev_day_peak captures the maximum output from 48 hours ago, indicating recent clear-sky potential.  

hour_avg provides the historical mean production at each hour, giving the model a site-specific baseline expectation.  

day_index tracks the progression through the dataset, capturing the gradual decline in output (as observed in eda). 

In [515]:
df['minute'] = df['DATE_TIME'].dt.minute
df['hour'] = df['DATE_TIME'].dt.hour
df['month'] = df['DATE_TIME'].dt.month
df['dayofweek'] = df['DATE_TIME'].dt.dayofweek
df['day_of_month'] = df['DATE_TIME'].dt.day
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['hour_squared'] = (df['hour'] - 12) ** 2    # negative relationship
df['part_of_day'] = (df['DATE_TIME'].dt.hour // 4).astype(int)
df['is_daytime'] = df['DATE_TIME'].dt.hour.between(5, 19).astype(int)
df['day_index'] = (df['DATE_TIME'] - df['DATE_TIME'].min()).dt.days
df['prev_day_peak'] = df.groupby(df['DATE_TIME'].dt.date)['AC_POWER'].transform('max').shift(96)
df['hour_avg'] = df.groupby('hour')['AC_POWER'].expanding().mean().reset_index(level=0, drop=True).shift(96)

### Lag Features

For 48-hour-ahead forecasting, all features must use data available at prediction time. With 30-minute intervals, this means a minimum shift of 96 steps. Individual lags at 96, 144, and 192 steps (2, 3, and 4 days ago) were combined into means to reduce overfitting risk on the limited dataset:

ac_lag_mean, dc_lag_mean, irr_lag_mean, temp_lag_mean, module_temp_lag_mean — average production and weather conditions over the previous 2-4 days. These smoothed signals proved more robust than individual lags.

efficiency_lag_mean — average ratio of AC_POWER to IRRADIATION over recent days, capturing panel conversion efficiency trends.

daily_yield_lag_mean — average cumulative daily production from recent days, indicating overall site performance level.

ac_lag_trend — difference between 2-day-ago and 4-day-ago output, indicating whether production is improving or declining. A positive trend suggests clearing weather, negative suggests increasing cloud cover. (similar to day_index)

In [516]:
df['ac_lag_96'] = df['AC_POWER'].shift(96)   # 48 hours ago
df['ac_lag_144'] = df['AC_POWER'].shift(144)  # 72 hours (3 days) ago
df['ac_lag_192'] = df['AC_POWER'].shift(192)  # 96 hours (4 days) ago

df['dc_lag_96'] = df['DC_POWER'].shift(96)
df['dc_lag_144'] = df['DC_POWER'].shift(144)
df['dc_lag_192'] = df['DC_POWER'].shift(192)

df['irr_lag_96'] = df['IRRADIATION'].shift(96)
df['irr_lag_144'] = df['IRRADIATION'].shift(144)
df['irr_lag_192'] = df['IRRADIATION'].shift(192)

df['temp_lag_96'] = df['AMBIENT_TEMPERATURE'].shift(96)
df['temp_lag_144'] = df['AMBIENT_TEMPERATURE'].shift(144)
df['temp_lag_192'] = df['AMBIENT_TEMPERATURE'].shift(192)

df['daily_yield_lag_96'] = df['DAILY_YIELD'].shift(96)
df['daily_yield_lag_144'] = df['DAILY_YIELD'].shift(144)
df['daily_yield_lag_192'] = df['DAILY_YIELD'].shift(192)

df['efficiency_lag_96'] = (df['AC_POWER'] / df['IRRADIATION'].replace(0, np.nan)).shift(96)
df['efficiency_lag_144'] = (df['AC_POWER'] / df['IRRADIATION'].replace(0, np.nan)).shift(144)
df['efficiency_lag_192'] = (df['AC_POWER'] / df['IRRADIATION'].replace(0, np.nan)).shift(192)
df['efficiency_lag_96'] = df['efficiency_lag_96'].fillna(0)
df['efficiency_lag_144'] = df['efficiency_lag_144'].fillna(0)
df['efficiency_lag_192'] = df['efficiency_lag_192'].fillna(0)

df['module_temp_lag_96'] = df['MODULE_TEMPERATURE'].shift(96)
df['module_temp_lag_144'] = df['MODULE_TEMPERATURE'].shift(144)
df['module_temp_lag_192'] = df['MODULE_TEMPERATURE'].shift(192)

In [517]:
df['ac_lag_mean'] = df[['ac_lag_96', 'ac_lag_144', 'ac_lag_192']].mean(axis=1)
df['irr_lag_mean'] = df[['irr_lag_96', 'irr_lag_144', 'irr_lag_192']].mean(axis=1)
df['temp_lag_mean'] = df[['temp_lag_96', 'temp_lag_144', 'temp_lag_192']].mean(axis=1)
df['ac_lag_trend'] = df['ac_lag_96'] - df['ac_lag_192']
df['efficiency_lag_mean'] = df[['efficiency_lag_96', 'efficiency_lag_144', 'efficiency_lag_192']].mean(axis=1)
df['daily_yield_lag_mean'] = df[['daily_yield_lag_96', 'daily_yield_lag_144', 'daily_yield_lag_192']].mean(axis=1)
df['module_temp_lag_mean'] = df[['module_temp_lag_96', 'module_temp_lag_144', 'module_temp_lag_192']].mean(axis=1)
df['dc_lag_mean'] = df[['dc_lag_96', 'dc_lag_144', 'dc_lag_192']].mean(axis=1)

### Rolling Features

Rolling statistics are computed on data shifted by 96 steps (48 hours) to maintain the forecasting constraint. All use min_periods=1 to maximise available training data.

AC Power rolling stats — means at 48, 96, and 192 step windows (1, 2, and 4 days) capture recent production trends at different timescales. Standard deviations at matching windows capture output volatility — high values indicate unstable cloud cover. Rolling max and min over 192 steps capture the best and worst recent performance, with their range indicating the spread between clear and cloudy days.

Irradiation rolling stats — means and standard deviations at 48, 96, and 192 step windows provide a direct measure of recent sunlight conditions and their stability.

Temperature rolling stats — 192-step mean and standard deviation for both ambient and module temperature. 

In [518]:
df['ac_rolling_mean_48'] = df['AC_POWER'].shift(96).rolling(48, min_periods=1).mean()
df['ac_rolling_mean_96'] = df['AC_POWER'].shift(96).rolling(96, min_periods=1).mean()
df['ac_rolling_mean_192'] = df['AC_POWER'].shift(96).rolling(192, min_periods=1).mean()
df['ac_rolling_std_48'] = df['AC_POWER'].shift(96).rolling(48, min_periods=1).std()
df['ac_rolling_std_96'] = df['AC_POWER'].shift(96).rolling(96, min_periods=1).std()
df['ac_rolling_std_192'] = df['AC_POWER'].shift(96).rolling(192, min_periods=1).std()

# Rolling min/max
df['ac_rolling_max_192'] = df['AC_POWER'].shift(96).rolling(192, min_periods=1).max()
df['ac_rolling_min_192'] = df['AC_POWER'].shift(96).rolling(192, min_periods=1).min()

# Rolling range
df['ac_rolling_range_192'] = df['ac_rolling_max_192'] - df['ac_rolling_min_192']

df['irr_rolling_mean_48'] = df['IRRADIATION'].shift(96).rolling(48, min_periods=1).mean()
df['irr_rolling_mean_96'] = df['IRRADIATION'].shift(96).rolling(96, min_periods=1).mean()
df['irr_rolling_mean_192'] = df['IRRADIATION'].shift(96).rolling(192, min_periods=1).mean()
df['irr_rolling_std_48'] = df['IRRADIATION'].shift(96).rolling(48, min_periods=1).std()
df['irr_rolling_std_96'] = df['IRRADIATION'].shift(96).rolling(96, min_periods=1).std()
df['irr_rolling_std_192'] = df['IRRADIATION'].shift(96).rolling(192, min_periods=1).std()

df['module_temp_rolling_mean_192'] = df['MODULE_TEMPERATURE'].shift(96).rolling(192, min_periods=1).mean()
df['module_temp_rolling_std_192'] = df['MODULE_TEMPERATURE'].shift(96).rolling(192, min_periods=1).std()

df['temp_rolling_mean_192'] = df['AMBIENT_TEMPERATURE'].shift(96).rolling(192, min_periods=1).mean()
df['temp_rolling_std_192'] = df['AMBIENT_TEMPERATURE'].shift(96).rolling(192, min_periods=1).std()

### Additional Weather Features

solar_elevation and solar_azimuth are calculated using pvlib from the site's estimated coordinates (12°N, 78°E) and provide the actual sun position at each timestamp. These are known exactly for any future time.

In [519]:
times = pd.DatetimeIndex(df['DATE_TIME']).tz_localize('Asia/Kolkata')
solar_pos = get_solarposition(times, latitude=12.0, longitude=78.0)
df['solar_elevation'] = solar_pos['elevation'].clip(lower=0).values
df['solar_azimuth'] = solar_pos['azimuth'].values

clear_sky_ghi is the theoretical maximum solar irradiance under perfectly clear skies, calculated from the site's coordinates and timestamp using pvlib. It depends only on sun position and atmospheric physics.

clear_sky_index is the ratio of actual irradiation to clear sky GHI, effectively measuring cloudiness. A value of 1.0 means perfectly clear, 0.5 means clouds are blocking half the sunlight.

In [520]:
site = Location(12.0, 78.0)
clear_sky = site.get_clearsky(times)
df['clear_sky_ghi'] = clear_sky['ghi'].values
df['clear_sky_index'] = (df['IRRADIATION'] / df['clear_sky_ghi'].replace(0, np.nan)).clip(0, 1.5).fillna(0)
df['clear_sky_index_lag_96'] = df['clear_sky_index'].shift(96) # only use lags to avoid leakage
df['clear_sky_index_lag_144'] = df['clear_sky_index'].shift(144)
df['clear_sky_index_lag_192'] = df['clear_sky_index'].shift(192)
df['clear_sky_index_lag_mean'] = df[['clear_sky_index_lag_96', 'clear_sky_index_lag_144', 'clear_sky_index_lag_192']].mean(axis=1)

In [521]:
feature_cols = [
    'hour', 'month', 'dayofweek', 'hour_sin', 'hour_cos', 'hour_squared', 'part_of_day', 'is_daytime',
    'prev_day_peak', 'hour_avg',
    'ac_rolling_mean_48', 'ac_rolling_mean_96', 'ac_rolling_mean_192',
    'ac_rolling_std_48', 'ac_rolling_std_96', 'ac_rolling_std_192', 
    'irr_rolling_mean_48', 'irr_rolling_mean_96', 'irr_rolling_mean_192',
    'temp_rolling_mean_192', 'temp_rolling_std_192',
    'irr_rolling_std_48', 'irr_rolling_std_96', 'irr_rolling_std_192', 
    'ac_rolling_max_192', 'ac_rolling_min_192', 'ac_rolling_range_192',
    'solar_elevation', 'solar_azimuth',
    'clear_sky_ghi', 
    'ac_lag_mean', 'irr_lag_mean', 'temp_lag_mean', 'ac_lag_trend', 
    'clear_sky_index_lag_mean', 'efficiency_lag_mean', 'daily_yield_lag_mean', 'module_temp_lag_mean', 'dc_lag_mean', 
    'module_temp_rolling_mean_192', 'module_temp_rolling_std_192'
]

TARGET = 'AC_POWER'

# Drop rows with NaN from lags
df_clean = df.dropna(subset=feature_cols + [TARGET]).reset_index(drop=True)
print(f'Before: {len(df)} rows')
print(f'After: {len(df_clean)} rows (lost {len(df) - len(df_clean)} to lag warm-up)')

Before: 1632 rows
After: 1360 rows (lost 272 to lag warm-up)


### Save

In [522]:
df_clean.to_csv('../data/features_30min.csv', index=False)